# Module 1 — Part 2: Agents

Lessons 13–14: Function calling + Agentic loop.

**Key shift:** Fixed RAG always searches once with the exact user query.
An agent lets the LLM decide when to search, what to search for, and when to stop.

**Groq adaptation note:**
- Lesson uses OpenAI Responses API: `response.output`, `item.type == 'function_call'`, `function_call_output` message type
- Groq chat.completions: `response.choices[0].message.tool_calls`, tool results use `role: 'tool'`

## Setup

In [1]:
from dotenv import load_dotenv
import os
import json
load_dotenv()

from openai import OpenAI
from ingest import load_faq_data, build_index

groq_client = OpenAI(
    api_key=os.getenv("GROQ_API_KEY"),
    base_url="https://api.groq.com/openai/v1"
)

MODEL = "openai/gpt-oss-120b"

documents = load_faq_data()
index = build_index(documents)
print("Index ready")

Index ready


## Lesson 13 — Function Calling

Without tools the LLM answers from general knowledge — vague and unhelpful for course questions.

In [2]:
messages = [
    {"role": "user", "content": "I just discovered the course. Can I join it?"}
]

response = groq_client.chat.completions.create(
    model=MODEL,
    messages=messages,
)

print(response.choices[0].message.content)
# Generic answer — no course-specific knowledge

Absolutely! I’d be happy to help you get started. Could you let me know which course you’re interested in joining? If you have a specific title or subject in mind, just share it and I’ll walk you through the enrollment steps.


### Define the search tool

We describe `search` in JSON schema. The LLM reads the `description` to decide when to call it.

In [3]:
def search(query):
    return index.search(
        query,
        num_results=5,
        boost_dict={"question": 3.0, "section": 0.5},
        filter_dict={"course": "llm-zoomcamp"}
    )

# Tool schema — chat.completions format (nested under 'function' key)
search_tool = {
    "type": "function",
    "function": {
        "name": "search",
        "description": "Search the FAQ database for entries matching the given query.",
        "parameters": {
            "type": "object",
            "properties": {
                "query": {
                    "type": "string",
                    "description": "Search query text to look up in the course FAQ."
                }
            },
            "required": ["query"],
            "additionalProperties": False
        }
    }
}

In [4]:
# Send question WITH the tool — LLM decides to call search instead of answering directly
messages = [
    {"role": "user", "content": "I just discovered the course. Can I join it?"}
]

response = groq_client.chat.completions.create(
    model=MODEL,
    messages=messages,
    tools=[search_tool],
)

assistant_message = response.choices[0].message
print("Tool calls:", assistant_message.tool_calls)
# Model returns a tool_call instead of a text answer

Tool calls: [ChatCompletionMessageFunctionToolCall(id='fc_150708fe-87c0-4771-96fa-fa4428776abc', function=Function(arguments='{"query":"Can I join the course"}', name='search'), type='function')]


In [5]:
# Execute the function and send result back
tool_call = assistant_message.tool_calls[0]
func_name = tool_call.function.name
args = json.loads(tool_call.function.arguments)

print(f"Function: {func_name}, Args: {args}")
# Note: model often rewrites the query to better keywords

results = search(**args)
result_json = json.dumps(results, indent=2)

# Build updated message history
# 1. append the assistant's tool-call message
# 2. append the tool result with role='tool'
messages.append(assistant_message)
messages.append({
    "role": "tool",
    "tool_call_id": tool_call.id,
    "content": result_json,
})

# Second API call — model now has the search results and generates final answer
response2 = groq_client.chat.completions.create(
    model=MODEL,
    messages=messages,
    tools=[search_tool],
)

print(response2.choices[0].message.content)

Function: search, Args: {'query': 'Can I join the course'}


Yes—you can still join the LLM Zoomcamp. Just sign up and start working through the material.  

If you’d like to earn a certificate, be sure to submit your capstone project (and complete any required peer reviews) before the current cohort’s submission window closes. Happy learning!


## Lesson 14 — The Agentic Loop

Single function call works for one turn. The loop handles:
- Multiple searches per question
- Model recovering from bad results (e.g. typos)
- Unknown number of tool calls in advance

In [6]:
def make_call(tool_call):
    """Execute a tool call and return the tool result message."""
    args = json.loads(tool_call.function.arguments)

    if tool_call.function.name == "search":
        result = search(**args)

    return {
        "role": "tool",
        "tool_call_id": tool_call.id,
        "content": json.dumps(result, indent=2),
    }

In [7]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function.
Use as many keywords from the user question as possible when making first requests.

Make multiple searches. First perform search, analyze the results
and then perform more searches.

The question has to be about the course or its logistics, off-topic questions
shouldn't be answered. If the search returns nothing, it's likely an off-topic question.
If you can't answer the question using FAQ, don't do it yourself. Only use the
facts from the FAQ database.

At the end, ask if there are other areas that the user wants to explore.
""".strip()

In [8]:
def agent_loop(instructions, question, model=MODEL) -> str:
    messages = [
        {"role": "system", "content": instructions},  # 'developer' in lesson -> 'system' for chat.completions
        {"role": "user", "content": question}
    ]

    it = 1
    last_answer = ""

    while True:
        print(f"iteration #{it}...")

        response = groq_client.chat.completions.create(
            model=model,
            messages=messages,
            tools=[search_tool]
        )

        assistant_message = response.choices[0].message
        messages.append(assistant_message)

        has_tool_calls = bool(assistant_message.tool_calls)

        if has_tool_calls:
            for tool_call in assistant_message.tool_calls:
                print(f"  function_call: {tool_call.function.name} {tool_call.function.arguments}")
                call_output = make_call(tool_call)
                messages.append(call_output)
        else:
            last_answer = assistant_message.content
            print("ASSISTANT:")
            print(last_answer)

        it += 1
        if not has_tool_calls:
            break

    return last_answer

In [9]:
# Typo test. In the lesson (OpenAI model) the agent searches 'Olama', gets bad
# results, then corrects itself to 'Ollama'. gpt-oss-120b retried three
# 'Olama' variants without fixing the typo, then admitted the FAQ had no answer:
# grounded (no hallucination) but less resourceful. Self-correction is model-dependent.
agent_loop(instructions, "How do I run Olama locally?")

iteration #1...


  function_call: search {"query":"Olama run locally"}
iteration #2...


  function_call: search {"query":"Olama"}
iteration #3...


  function_call: search {"query":"run Olama"}
iteration #4...


ASSISTANT:
I’m sorry—I couldn’t find any FAQ entry that explains how to run **Olama** locally. The course FAQ only includes a general entry about running the whole course locally (see “Can I run the course locally instead of Codespaces?”), but it doesn’t contain specific instructions for Olama.

If you need help with Olama, I recommend checking the official Olama documentation or the repository’s README for setup instructions. If you have any other questions about the course material, logistics, or the FAQs, feel free to let me know!


'I’m sorry—I couldn’t find any FAQ entry that explains how to run **Olama** locally. The course FAQ only includes a general entry about running the whole course locally (see “Can I run the course locally instead of Codespaces?”), but it doesn’t contain specific instructions for Olama.\n\nIf you need help with Olama, I recommend checking the official Olama documentation or the repository’s README for setup instructions. If you have any other questions about the course material, logistics, or the FAQs, feel free to let me know!'

In [10]:
agent_loop(instructions, "I just discovered the course. Can I still join it?")

iteration #1...


  function_call: search {"query":"late enrollment can I still join course"}
iteration #2...


ASSISTANT:
Yes—you can still join the course. If you’d like to earn a certificate, just be sure to submit your capstone project before the submission window closes (the form stays open for a while after the official deadline).  

Is there anything else about the course you’d like to explore?


'Yes—you can still join the course.\u202fIf you’d like to earn a certificate, just be sure to submit your capstone project before the submission window closes (the form stays open for a while after the official deadline).  \n\nIs there anything else about the course you’d like to explore?'

In [11]:
# Off-topic guardrail test — agent should refuse
agent_loop(instructions, "What's the Queen's Gambit?")

iteration #1...


  function_call: search {"query":"Queen's Gambit"}
iteration #2...


ASSISTANT:
I’m sorry, but I don’t have information on that topic in the course FAQ. Is there anything else related to the course that I can help you with?


'I’m sorry, but I don’t have information on that topic in the course FAQ. Is there anything else related to the course that I can help you with?'